# 33. Square-Dalitz SCF migration maps (dense)

**Objectives:**
- Build a `SquareDalitzSCFMap` with a Gaussian-smearing migration kernel in
  `(m', theta')` Square-Dalitz coordinates, stored densely.
- Query `scf_fraction_at` and `smeared_density_at` on real Dalitz points.
- Use the map inside an `SCFSignalPDF` and run one small toy fit.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero. Self-cross-feed (SCF) is treated as part of the
signal model here, not as an incoherent background; see `docs/scf.md`.

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

import numpy as np
import jax.numpy as jnp
from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, Parameter, RealImag, Resonance,
    SCFSignalPDF, SquareDalitzSCFMap, generate_toy,
)
from dalitzplotfitter.integration import GridIntegrator

## 1. A small model

`B+ -> K+ pi+ pi-` again, with a `K*(892)` resonance and a non-resonant term.
We use the Square-Dalitz normalization grid so `model.normalization_sample`
and the SCF map's own `(m', theta')` binning share the same `pair`.

In [2]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))
nr_x = Parameter.coefficient("NR.x", 0.4, owner="NR", bounds=(-2, 2))
nr_y = Parameter.coefficient("NR.y", -0.2, owner="NR", bounds=(-2, 2))
model = DecayModel(
    channel,
    [
        Resonance("Kstar", (0, 2), RealImag(1.0, 0.0), mass=0.892, width=0.051, spin=1),
        NonResonant(RealImag(nr_x, nr_y), name="NR"),
    ],
    normalization_method="square-dalitz", normalization_pair=(0, 2),
    normalization_resolution=60,
)
truth = {p.name: p.value for p in model.parameters}

## 2. A Gaussian-smearing migration matrix

Following the Laura++ `LauScfMap` convention (`docs/scf.md`): every true bin's
row is a normalized probability distribution over reconstructed bins,
`sum_r M[t, r] = 1`. We build a simple tridiagonal-in-flattened-index smearing
kernel (most probability stays in the true bin, some migrates to its
neighbors) and normalize each row to sum to one. `bins_mprime = bins_thetaprime
= 10` keeps the dense `n_bins x n_bins` matrix tiny (100 x 100) for this
tutorial.

In [3]:
N = 10
n_bins = N * N

migration = np.eye(n_bins) * 0.70
migration += np.roll(np.eye(n_bins), 1, axis=1) * 0.15
migration += np.roll(np.eye(n_bins), -1, axis=1) * 0.15
migration /= migration.sum(axis=1, keepdims=True)

scf_fraction = np.full(n_bins, 0.12)

scf_map = SquareDalitzSCFMap(
    migration=migration,
    scf_fraction=scf_fraction,
    mother_mass=channel.parent_mass,
    masses=channel.daughter_masses,
    bins_mprime=N,
    bins_thetaprime=N,
    pair=(0, 2),
    storage="dense",
)
print(f"is_sparse={scf_map.is_sparse}, migration_nnz={scf_map.migration_nnz}, "
      f"migration_density={scf_map.migration_density:.3f}")

is_sparse=False, migration_nnz=300, migration_density=0.030


## 3. `scf_fraction_at` and `smeared_density_at`

`scf_fraction_at` looks up the true-bin SCF fraction for arbitrary event
invariants; here it is uniform (0.12) everywhere by construction.
`smeared_density_at` migrates a true density (evaluated at the map's own
bin centres, `scf_map.true_bin_data()`) and samples the migrated
reconstructed density back at the requested invariants.

In [4]:
toy = generate_toy(model, 4000, parameters=truth, seed=33, include_momenta=False)
data = toy.as_dict()

fractions = np.asarray(scf_map.scf_fraction_at(data["s12"], data["s13"], data["s23"]))
print(f"SCF fraction at toy events: min={fractions.min():.3f}, max={fractions.max():.3f}")

true_data = scf_map.true_bin_data()
true_density = model.intensity(true_data, truth)
smeared = np.asarray(
    scf_map.smeared_density_at(true_density, data["s12"], data["s13"], data["s23"])
)
print(f"Smeared SCF density at toy events: min={smeared.min():.4f}, mean={smeared.mean():.4f}")
assert np.all(smeared >= 0.0)

SCF fraction at toy events: min=0.120, max=0.120


Smeared SCF density at toy events: min=0.0000, mean=0.0465


## 4. `SCFSignalPDF` and one small toy fit

`SCFSignalPDF` combines the correctly-reconstructed part `(1 - f_SCF) * intensity`
with the migrated SCF part into one normalized reconstructed-space density
(`docs/scf.md`). We float the non-resonant coefficient and run a short Minuit
fit with `iminuit` directly (the low-level API, since `SCFSignalPDF` is not
yet wrapped by `FitSession`).

In [5]:
from iminuit import Minuit

pdf = SCFSignalPDF(
    intensity=lambda d, p: model.intensity(d, p),
    integrator=GridIntegrator(model.normalization_sample),
    scf_map=scf_map,
)

def nll(nr_x, nr_y):
    parameters = dict(truth)
    parameters["NR.x"] = nr_x
    parameters["NR.y"] = nr_y
    return -float(jnp.sum(pdf.logpdf(data, parameters)))

start_x, start_y = 0.25, -0.05  # deliberately off from the truth (0.4, -0.2)
minuit = Minuit(nll, nr_x=start_x, nr_y=start_y)
minuit.errordef = Minuit.LIKELIHOOD
minuit.limits["nr_x"] = (-2, 2)
minuit.limits["nr_y"] = (-2, 2)
minuit.migrad(ncall=2000)
print(minuit.values)
assert minuit.valid

<ValueView nr_x=0.48459726370702444 nr_y=-0.23340803753723915>


## Try it yourself

1. Compare `scf_map.migration_matrix()` to the input `migration` array.
2. Make the smearing kernel wider (three or five diagonals) and see how
   `smeared_density_at` changes.
3. Add a reconstructed-space `veto=` to `SCFSignalPDF` and check `pdf.normalization`
   changes (see `docs/scf.md`, "Reconstructed-space vetoes").

## Continue learning

Next: [Sparse SCF migration](tutorial_34_sparse_migration.ipynb).
Reference: [SCF migration](../../docs/scf.md).

Return to [the course guide](TUTORIALS.md).